# Flood Risk v2 — U-Net Colab Runner
**ESCP Hackathon 2026 · Group 2 · Train: Severn → Test: Northumbria**

Replaces v1 XGBoost ordinal ensemble (31.21% Macro F1) with a multi-task U-Net.  
Target: **45–55% Macro F1**.

Runtime: ~2–4 hours total on Colab Pro GPU.

Steps:
1. Mount Drive, clone repo, install dependencies
2. Build raster cache (one-off, ~10 min)
3. Train U-Net (~1–3 hours)
4. Inference + metrics + maps (~10 min)

---
## Step 1 — Mount Drive & set up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

# IMPORTANT: change this to your actual data folder if different
DRIVE_DATA = '/content/drive/MyDrive/Hackathon II Group 2'

assert os.path.exists(DRIVE_DATA), f'Data folder not found: {DRIVE_DATA}'
print('Data folder contents:')
for f in sorted(os.listdir(DRIVE_DATA)):
    p = os.path.join(DRIVE_DATA, f)
    if os.path.isfile(p):
        size = os.path.getsize(p)
        label = f'{size/1e9:.2f} GB' if size > 1e6 else f'{size} B'
        print(f'  {f:<45} {label}')

In [ ]:
# Clone the repo (or pull latest)
REPO_URL = 'https://github.com/victorbomberna3/flood-risk-v2.git'   # update to your repo URL
REPO_DIR = '/content/flood-risk-v2'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

---
## Step 2 — Build raster cache

Loads .nc files, engineers features (HAND, TWI, slope, aspect, curvature, weather rolling stats), and saves as compressed .npz arrays.

**Run once.** Subsequent training runs skip this step (~10 min total for both regions).

In [ ]:
# Patch config with the actual Drive data path (in case you changed it above)
import yaml
with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['drive_data'] = DRIVE_DATA
with open('configs/default.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Config updated with:', cfg['paths']['drive_data'])

In [ ]:
!python scripts/01_build_rasters.py --config configs/default.yaml

---
## Step 3 — Train U-Net

30 epochs × ~15 batches/epoch. Mixed-precision on A100/V100/T4.

Watch the `val_macroF1` line per epoch. Should rise from ~0.20 at epoch 1 to 0.40+ by epoch 10–15.

In [ ]:
!python scripts/02_train.py --config configs/default.yaml

---
## Step 4 — Inference on Northumbria

Sliding-window prediction with 50% overlap + 8-way test-time augmentation (TTA).

Outputs:
- `outputs/predictions_northumbria.parquet` — class predictions per pixel
- `outputs/metrics.json` — Macro F1, per-class F1, confusion matrices
- `outputs/maps/*.png` — prediction vs ground-truth comparison maps

In [ ]:
!python scripts/03_predict.py --config configs/default.yaml

---
## Step 5 — Inspect results

In [ ]:
# Display prediction vs ground-truth maps
from IPython.display import Image, display
for f in sorted(os.listdir('outputs/maps')):
    print(f'\n=== {f} ===')
    display(Image(f'outputs/maps/{f}'))

In [ ]:
# Final metrics summary
import json
with open('outputs/metrics.json') as f:
    m = json.load(f)

print(f"Mean Macro F1 (across 5 depths): {m['macro_f1_mean']:.4f}")
print()
for tname in ['risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m']:
    f1 = m[tname]['macro_f1']
    per_class = m[tname]['per_class_f1']
    print(f"  {tname}  Macro F1 = {f1:.4f}")
    print(f"    Per-class F1: {[f'{x:.3f}' for x in per_class]}")

In [ ]:
# Copy outputs back to Drive for safekeeping
import shutil
dest = f'{DRIVE_DATA}/outputs_v2'
os.makedirs(dest, exist_ok=True)
for item in ['predictions_northumbria.parquet', 'metrics.json', 'maps']:
    src = f'outputs/{item}'
    dst = f'{dest}/{item}'
    if os.path.isdir(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'  Copied → {dst}')

# Also copy the trained model
shutil.copy2('checkpoints/model_best.pt', f'{dest}/model_best.pt')
print(f'  Copied → {dest}/model_best.pt')

---
## Iteration tips

If results are below 45% Macro F1 after first run:

**Most impactful tweaks** (in order to try):
1. Increase `training.epochs` from 30 → 50 in `configs/default.yaml`
2. Try `loss.type: focal` with `focal_gamma: 2.0`
3. Reduce `model.base_channels` from 32 → 24 if overfitting (large train-val gap)
4. Add weather features: edit `configs/default.yaml` features.weather list

**To investigate further:**
- Look at `checkpoints/history.json` for training curves
- Inspect confusion matrices — if class 4 (High) F1 is very low, weight it more heavily in `loss.class_weight_method` (try `effective_num`)
- If predictions look 'striped' or 'patchy' in the maps, the patch overlap stride is too large; reduce `patches.stride_inference` to 64